# 06 — Convergence, ECDFs and Strategy Ablations

Run independently of notebooks 05 and 07. These exploratory adaptive-target plots
are separate from fixed-target headline reliability. High-resolution PNG sweeps
can take time on their first run: execute only the sections you need, or set the
filters below. Only calculated statistics are saved; PNG exports are not cached.
Profiles and AUC use the configured objective-call allowance: currently a fixed 100,000 evaluations at every dimension. Saved outputs and saved numerical snapshots may describe the previous protocol; recalculate only after matching new evaluations are available.
Evaluations with a different recorded budget are excluded from the current analysis.
After evaluation, recalculate profiles and performance before exporting; old snapshots are historical.
No inline images or HTML exports.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from benchmarking.infra.di import analysis_scope
from benchmarking.application.benchmark_config import BenchmarkConfig, ReliabilityConfig
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter


figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None


analysis_config = BenchmarkConfig(
    budget=100_000, trials_per_condition=20, random_seed=42, first_instance_id=2,
    trial_timeout_seconds=30.0, observation_trace_points=64,
    classical_baselines=["cmaes", "de", "pso"],
    noise_robustness_enabled=True, cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    reliability=ReliabilityConfig(
        primary_prompt_strategy="baseline", primary_target=1e-8, secondary_target=1e-2,
        bootstrap_samples=1000, bootstrap_seed=20260923,
        checkpoint_fractions=[0.01, 0.1, 0.5, 1.0],
    ),
)

# Notebook resource lifetime (rerun-safe; no workflow is launched here).
import atexit
import os
from contextlib import ExitStack
from functools import partial


def close_notebook_resources(resources: ExitStack, owner_pid: int) -> None:
    if os.getpid() == owner_pid:
        resources.close()


if "close_analysis" in globals():
    atexit.unregister(close_analysis)
    close_analysis()
analysis_resources = ExitStack()
close_analysis = partial(close_notebook_resources, analysis_resources, os.getpid())
try:
    analysis = analysis_resources.enter_context(analysis_scope(analysis_config))
except BaseException:
    close_analysis()
    raise
atexit.register(close_analysis)


In [2]:
performance = analysis.performance.execute(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
performance_snapshot = analysis.results_store.save(performance)
print(f"Calculated exploratory adaptive-target metrics for {len(performance.table)} entries.")

print(f"Saved numerical results: {performance_snapshot}")


Calculated exploratory adaptive-target metrics for 11 entries.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/performance/3ea6a671129d221f834a9140200e4bdf48163891a1e6011741a8d813bbd79eff


## Exploratory model comparison


In [3]:
figure_exporter.export_model_scale(performance)


## Strategy ablation by landscape hardness


In [4]:
figure_exporter.export_hardness_ablation(performance)


## Explicit convergence and ECDF profiles

Optional high-resolution sweep; run this cell only when needed.


In [5]:
explicit_profiles = analysis.ecdf_and_convergence.execute(mode="explicit", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
explicit_profiles_snapshot = analysis.results_store.save(explicit_profiles)
print(f"Calculated {len(explicit_profiles.conditions)} explicit condition profiles; no PNGs exported yet.")

print(f"Saved numerical results: {explicit_profiles_snapshot}")


Calculated 1 explicit condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/explicit/d317009f13d24139bbd8af54f6be4f1cc06765807dd4874d76ccedc2b2d9b646


In [6]:
figure_exporter.export_profiles(explicit_profiles)


## Implicit convergence and ECDF profiles

Optional high-resolution sweep; run this cell only when needed.


In [7]:
implicit_profiles = analysis.ecdf_and_convergence.execute(mode="implicit", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
implicit_profiles_snapshot = analysis.results_store.save(implicit_profiles)
print(f"Calculated {len(implicit_profiles.conditions)} implicit condition profiles; no PNGs exported yet.")

print(f"Saved numerical results: {implicit_profiles_snapshot}")


Calculated 1 implicit condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/implicit/c13d74174b11c05aa36b61b16f420febc4f106109f4522632499faebb5d801a0


In [8]:
figure_exporter.export_profiles(implicit_profiles)


## Explicit versus implicit strategy comparison

Optional high-resolution sweep; run this cell only when needed.


In [9]:
comparison_profiles = analysis.ecdf_and_convergence.execute(mode="comparison", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
comparison_profiles_snapshot = analysis.results_store.save(comparison_profiles)
print(f"Calculated {len(comparison_profiles.conditions)} comparison condition profiles; no PNGs exported yet.")

print(f"Saved numerical results: {comparison_profiles_snapshot}")


Calculated 1 comparison condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/comparison/a1ee6d2db3c7687ae7939b395d94c886f6b18dca380a2e63c52aa4029dc9bda9


In [10]:
figure_exporter.export_profiles(comparison_profiles)
